**DATASET**

In [1]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="lxjOP157H1bZgHOniuFX")

ds1 = rf.workspace("counterfeitdrug").project("counterfeit-drugs-2.0-rros5").version(19).download("coco")
ds2 = rf.workspace("drkehowler-yahoo-com").project("blister-packaged-medicine-thesis").version(3).download("coco")

path1, path2 = ds1.location, ds2.location

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 361.2/361.2 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 73.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.4/58.4 kB 3.2 MB/s eta 0:00:00
  Attempting uninstall: typer
    Found existing installation: typer 0.27.2
    Uninstalling typer-0.27.2:
      Successfully uninstalled typer-0.27.2
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Counterfeit-Drugs-2.0-19 in coco:: 100%|██████████| 3013/3013 [00:00<00:00, 5364.09it/s]


loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Blister-Packaged-Medicine-Thesis-3 in coco:: 100%|██████████| 1364/1364 [00:00<00:00, 4539.83it/s]


In [2]:
import json
for p in (path1, path2):
    cats = json.load(open(f"{p}/train/_annotations.coco.json"))["categories"]
    print(p, [(c["id"], c["name"]) for c in cats])

/kaggle/working/Counterfeit-Drugs-2.0-19 [(0, 'counterfeit-drugs-7j5D'), (1, 'authentic'), (2, 'counterfeit')]
/kaggle/working/Blister-Packaged-Medicine-Thesis-3 [(0, 'Medicine'), (1, 'authentic'), (2, 'counterfeit')]


In [3]:
import os, json, shutil

OUT = "merged_dataset"
SOURCES = {"ds1": path1, "ds2": path2}

# parent/junk categories to drop (lowercase), check the output of Step 2
IGNORE = {"counterfeit-drugs-7j5d","medicine"}   

# map differing names to one unified name, e.g. {"blister pack": "blister"}
RENAME = {}

def norm(name):
    name = name.strip().lower()
    return RENAME.get(name, name)

# build unified category list across all splits and datasets
names = set()
for p in SOURCES.values():
    for split in ("train", "valid", "test"):
        f = f"{p}/{split}/_annotations.coco.json"
        if os.path.exists(f):
            for c in json.load(open(f))["categories"]:
                if c["name"].lower() not in IGNORE:
                    names.add(norm(c["name"]))

unified = {n: i + 1 for i, n in enumerate(sorted(names))}  # ids start at 1
categories = [{"id": i, "name": n} for n, i in unified.items()]
print("Unified classes:", unified)

for split in ("train", "valid", "test"):
    out_dir = f"{OUT}/{split}"
    os.makedirs(out_dir, exist_ok=True)

    merged = {"images": [], "annotations": [], "categories": categories}
    img_id, ann_id = 1, 1

    for tag, p in SOURCES.items():
        f = f"{p}/{split}/_annotations.coco.json"
        if not os.path.exists(f):
            continue
        coco = json.load(open(f))

        cat_map = {}
        for c in coco["categories"]:
            if c["name"].lower() not in IGNORE:
                cat_map[c["id"]] = unified[norm(c["name"])]

        id_map = {}
        for im in coco["images"]:
            new_name = f"{tag}_{im['file_name']}"
            shutil.copy(f"{p}/{split}/{im['file_name']}", f"{out_dir}/{new_name}")
            id_map[im["id"]] = img_id
            merged["images"].append({**im, "id": img_id, "file_name": new_name})
            img_id += 1

        for a in coco["annotations"]:
            if a["category_id"] not in cat_map:
                continue
            merged["annotations"].append({
                **a,
                "id": ann_id,
                "image_id": id_map[a["image_id"]],
                "category_id": cat_map[a["category_id"]],
            })
            ann_id += 1

    json.dump(merged, open(f"{out_dir}/_annotations.coco.json", "w"))
    print(split, len(merged["images"]), "images,", len(merged["annotations"]), "annotations")

Unified classes: {'authentic': 1, 'counterfeit': 2}
train 3639 images, 3772 annotations
valid 336 images, 342 annotations
test 389 images, 395 annotations


In [4]:
import os
import json
import torch
import torchvision.transforms.v2

from torch.utils.data import Dataset
from torchvision import tv_tensors
from torchvision.io import read_image


class VOCDataset(Dataset):

    def __init__(self, split, dataset_path, im_size=300,
                 ignore_classes=("counterfeit-drugs-7j5D","Medicine")):

        self.split = split
        self.dataset_path = dataset_path
        self.im_size = im_size

        self.im_mean = [123.0, 117.0, 104.0]

        self.imagenet_mean = [
            0.485,
            0.456,
            0.406
        ]

        self.imagenet_std = [
            0.229,
            0.224,
            0.225
        ]

        annotation_file = os.path.join(
            dataset_path,
            split,
            "_annotations.coco.json"
        )

        if not os.path.exists(annotation_file):
            raise FileNotFoundError(
                f"Annotation file not found:\n{annotation_file}"
            )

        with open(annotation_file, "r") as f:
            coco = json.load(f)

        ignore = {name.lower() for name in ignore_classes}
        
        categories = sorted(
            [c for c in coco["categories"] if c["name"].lower() not in ignore],
            key=lambda x: x["id"]
        )

        self.classes = ["background"]

        for category in categories:
            self.classes.append(category["name"])

        self.label2idx = {
            name: idx
            for idx, name in enumerate(self.classes)
        }

        self.idx2label = {
            idx: name
            for idx, name in enumerate(self.classes)
        }

        self.category2idx = {
            category["id"]: self.label2idx[category["name"]]
            for category in categories
        }

        print("\nClasses:")
        for idx, name in self.idx2label.items():
            print(f"{idx}: {name}")

        annotations = {}

        for ann in coco["annotations"]:

            image_id = ann["image_id"]

            if image_id not in annotations:
                annotations[image_id] = []

            category_id = ann["category_id"]

            if category_id not in self.category2idx:
                continue

            x, y, w, h = ann["bbox"]

            if w <= 0 or h <= 0:
                continue

            annotations[image_id].append({
                "label": self.category2idx[category_id],
                "bbox": [
                    x,
                    y,
                    x + w,
                    y + h
                ],
                "difficult": 0
            })

        self.images_info = []

        split_dir = os.path.join(
            dataset_path,
            split
        )

        for image in coco["images"]:

            image_id = image["id"]

            filename = os.path.join(
                split_dir,
                image["file_name"]
            )

            if not os.path.exists(filename):
                continue

            detections = annotations.get(
                image_id,
                []
            )

            if len(detections) == 0:
                continue

            self.images_info.append({
                "img_id": image_id,
                "filename": filename,
                "width": image["width"],
                "height": image["height"],
                "detections": detections
            })

        print(
            f"\n{split} images: {len(self.images_info)}"
        )

        self.transforms = {

            "train": torchvision.transforms.v2.Compose([

                torchvision.transforms.v2.RandomPhotometricDistort(),

                torchvision.transforms.v2.RandomZoomOut(
                    fill=self.im_mean
                ),

                torchvision.transforms.v2.RandomIoUCrop(),

                torchvision.transforms.v2.RandomHorizontalFlip(
                    p=0.5
                ),

                torchvision.transforms.v2.Resize(
                    size=(self.im_size, self.im_size)
                ),

                torchvision.transforms.v2.SanitizeBoundingBoxes(
                    labels_getter=lambda x: (
                        x[1]["labels"],
                        x[1]["difficult"]
                    )
                ),

                torchvision.transforms.v2.ToPureTensor(),

                torchvision.transforms.v2.ToDtype(
                    torch.float32,
                    scale=True
                ),

                torchvision.transforms.v2.Normalize(
                    mean=self.imagenet_mean,
                    std=self.imagenet_std
                )
            ]),

            "valid": torchvision.transforms.v2.Compose([

                torchvision.transforms.v2.Resize(
                    size=(self.im_size, self.im_size)
                ),

                torchvision.transforms.v2.ToPureTensor(),

                torchvision.transforms.v2.ToDtype(
                    torch.float32,
                    scale=True
                ),

                torchvision.transforms.v2.Normalize(
                    mean=self.imagenet_mean,
                    std=self.imagenet_std
                )
            ]),

            "test": torchvision.transforms.v2.Compose([

                torchvision.transforms.v2.Resize(
                    size=(self.im_size, self.im_size)
                ),

                torchvision.transforms.v2.ToPureTensor(),

                torchvision.transforms.v2.ToDtype(
                    torch.float32,
                    scale=True
                ),

                torchvision.transforms.v2.Normalize(
                    mean=self.imagenet_mean,
                    std=self.imagenet_std
                )
            ])
        }

    def __len__(self):
        return len(self.images_info)

    def __getitem__(self, index):

        im_info = self.images_info[index]

        im = read_image(
            im_info["filename"]
        )

        if im.shape[0] == 1:
            im = im.repeat(3, 1, 1)

        elif im.shape[0] > 3:
            im = im[:3]

        boxes = [
            detection["bbox"]
            for detection in im_info["detections"]
        ]

        labels = [
            detection["label"]
            for detection in im_info["detections"]
        ]

        difficult = [
            detection["difficult"]
            for detection in im_info["detections"]
        ]

        targets = {}

        targets["bboxes"] = tv_tensors.BoundingBoxes(
            boxes,
            format="XYXY",
            canvas_size=im.shape[-2:]
        )

        targets["labels"] = torch.tensor(
            labels,
            dtype=torch.long
        )

        targets["difficult"] = torch.tensor(
            difficult,
            dtype=torch.long
        )

        im_tensor, targets = self.transforms[
            self.split
        ](
            im,
            targets
        )

        h, w = im_tensor.shape[-2:]

        wh_tensor = torch.tensor(
            [w, h, w, h],
            dtype=targets["bboxes"].dtype
        ).expand_as(
            targets["bboxes"]
        )

        targets["bboxes"] = (
            targets["bboxes"] / wh_tensor
        )

        return (
            im_tensor,
            targets,
            im_info["filename"]
        )

In [5]:
train_ds = VOCDataset("train", "merged_dataset", ignore_classes=("Medicine"))
val_ds   = VOCDataset("valid", "merged_dataset", ignore_classes=("Medicine"))
test_ds  = VOCDataset("test",  "merged_dataset", ignore_classes=("Medicine"))


Classes:
0: background
1: authentic
2: counterfeit

train images: 3639

Classes:
0: background
1: authentic
2: counterfeit

valid images: 336

Classes:
0: background
1: authentic
2: counterfeit

test images: 389


In [6]:
config = {
    "dataset_params": {
        "dataset_path": "merged_dataset",
        "num_classes": 3,
        "im_size": 300,
    },

    "model_params": {
        "im_channels": 3,
        "aspect_ratios": [
            [1.0, 2.0, 0.5],
            [1.0, 2.0, 3.0, 0.5, 0.333],
            [1.0, 2.0, 3.0, 0.5, 0.333],
            [1.0, 2.0, 3.0, 0.5, 0.333],
            [1.0, 2.0, 0.5],
            [1.0, 2.0, 0.5]
        ],
        "scales": [
            0.1,
            0.2,
            0.375,
            0.55,
            0.725,
            0.9
        ],
        "iou_threshold": 0.5,
        "low_score_threshold": 0.01,
        "neg_pos_ratio": 3,
        "pre_nms_topK": 400,
        "detections_per_img": 200,
        "nms_threshold": 0.45,
    },

    "train_params": {
        "task_name": "voc",
        "seed": 1111,
        "acc_steps": 1,
        "num_epochs": 2,
        "batch_size": 8,
        "lr_steps": [40, 50, 60, 70, 80, 90],
        "lr": 0.001,
        "log_steps": 100,
        "infer_conf_threshold": 0.5,
        "ckpt_name": "ssd_final_aimedisina.pth",
    }
}

**MODEL**

In [7]:
import torch.nn as nn
import torch
import math
import torchvision


def get_iou(boxes1, boxes2):
    r"""
    IOU between two sets of boxes
    :param boxes1: (Tensor of shape N x 4)
    :param boxes2: (Tensor of shape M x 4)
    :return: IOU matrix of shape N x M
    """

    # Area of boxes (x2-x1)*(y2-y1)
    area1 = (boxes1[:, 2] - boxes1[:, 0]) * (boxes1[:, 3] - boxes1[:, 1])  # (N,)
    area2 = (boxes2[:, 2] - boxes2[:, 0]) * (boxes2[:, 3] - boxes2[:, 1])  # (M,)

    # Get top left x1,y1 coordinate
    x_left = torch.max(boxes1[:, None, 0], boxes2[:, 0])  # (N, M)
    y_top = torch.max(boxes1[:, None, 1], boxes2[:, 1])  # (N, M)

    # Get bottom right x2,y2 coordinate
    x_right = torch.min(boxes1[:, None, 2], boxes2[:, 2])  # (N, M)
    y_bottom = torch.min(boxes1[:, None, 3], boxes2[:, 3])  # (N, M)

    intersection_area = ((x_right - x_left).clamp(min=0) *
                         (y_bottom - y_top).clamp(min=0))  # (N, M)
    union = area1[:, None] + area2 - intersection_area  # (N, M)
    iou = intersection_area / union  # (N, M)
    return iou


def boxes_to_transformation_targets(ground_truth_boxes,
                                    default_boxes,
                                    weights=(10., 10., 5., 5.)):
    r"""
    Method to compute targets for each default_boxes.
    Assumes boxes are in x1y1x2y2 format.
    We first convert boxes to cx,cy,w,h format and then
    compute targets based on following formulation
    target_dx = (gt_cx - default_boxes_cx) / default_boxes_w
    target_dy = (gt_cy - default_boxes_cy) / default_boxes_h
    target_dw = log(gt_w / default_boxes_w)
    target_dh = log(gt_h / default_boxes_h)
    :param ground_truth_boxes: (Tensor of shape N x 4)
    :param default_boxes: (Tensor of shape N x 4)
    :param weights: Tuple[float] -> (wx, wy, ww, wh)
    :return: regression_targets: (Tensor of shape N x 4)
    """
    # # Get center_x,center_y,w,h from x1,y1,x2,y2 for default_boxes
    widths = default_boxes[:, 2] - default_boxes[:, 0]
    heights = default_boxes[:, 3] - default_boxes[:, 1]
    center_x = default_boxes[:, 0] + 0.5 * widths
    center_y = default_boxes[:, 1] + 0.5 * heights

    # # Get center_x,center_y,w,h from x1,y1,x2,y2 for gt boxes
    gt_widths = (ground_truth_boxes[:, 2] - ground_truth_boxes[:, 0])
    gt_heights = ground_truth_boxes[:, 3] - ground_truth_boxes[:, 1]
    gt_center_x = ground_truth_boxes[:, 0] + 0.5 * gt_widths
    gt_center_y = ground_truth_boxes[:, 1] + 0.5 * gt_heights

    # Use formulation to compute all targets
    targets_dx = weights[0] * (gt_center_x - center_x) / widths
    targets_dy = weights[1] * (gt_center_y - center_y) / heights
    targets_dw = weights[2] * torch.log(gt_widths / widths)
    targets_dh = weights[3] * torch.log(gt_heights / heights)
    regression_targets = torch.stack((targets_dx,
                                      targets_dy,
                                      targets_dw,
                                      targets_dh), dim=1)
    return regression_targets


def apply_regression_pred_to_default_boxes(box_transform_pred,
                                           default_boxes,
                                           weights=(10., 10., 5., 5.)):
    r"""
    Method to transform default_boxes based on transformation parameter
    prediction.
    Assumes boxes are in x1y1x2y2 format
    :param box_transform_pred: (Tensor of shape N x 4)
    :param default_boxes: (Tensor of shape N x 4)
    :param weights: Tuple[float] -> (wx, wy, ww, wh)
    :return: pred_boxes: (Tensor of shape N x 4)
    """

    # Get cx, cy, w, h from x1,y1,x2,y2
    w = default_boxes[:, 2] - default_boxes[:, 0]
    h = default_boxes[:, 3] - default_boxes[:, 1]
    center_x = default_boxes[:, 0] + 0.5 * w
    center_y = default_boxes[:, 1] + 0.5 * h

    dx = box_transform_pred[..., 0] / weights[0]
    dy = box_transform_pred[..., 1] / weights[1]
    dw = box_transform_pred[..., 2] / weights[2]
    dh = box_transform_pred[..., 3] / weights[3]
    # dh -> (num_default_boxes)

    pred_center_x = dx * w + center_x
    pred_center_y = dy * h + center_y
    pred_w = torch.exp(dw) * w
    pred_h = torch.exp(dh) * h
    # pred_center_x -> (num_default_boxes, 4)

    pred_box_x1 = pred_center_x - 0.5 * pred_w
    pred_box_y1 = pred_center_y - 0.5 * pred_h
    pred_box_x2 = pred_center_x + 0.5 * pred_w
    pred_box_y2 = pred_center_y + 0.5 * pred_h

    pred_boxes = torch.stack((
        pred_box_x1,
        pred_box_y1,
        pred_box_x2,
        pred_box_y2),
        dim=-1)
    return pred_boxes


def generate_default_boxes(feat, aspect_ratios, scales):
    r"""
    Method to generate default_boxes for all feature maps the image
    :param feat: List[(Tensor of shape B x C x Feat_H x Feat x W)]
    :param aspect_ratios: List[List[float]] aspect ratios for each feature map
    :param scales: List[float] scales for each feature map
    :return: default_boxes : List[(Tensor of shape N x 4)] default_boxes over all
            feature maps aggregated for each batch image
    """

    # List to store default boxes for all feature maps
    default_boxes = []
    for k in range(len(feat)):
        # We first add the aspect ratio 1 and scale (sqrt(scale[k])*sqrt(scale[k+1])
        s_prime_k = math.sqrt(scales[k] * scales[k + 1])
        wh_pairs = [[s_prime_k, s_prime_k]]

        # Adding all possible w,h pairs according to
        # aspect ratio of the feature map k
        for ar in aspect_ratios[k]:
            sq_ar = math.sqrt(ar)
            w = scales[k] * sq_ar
            h = scales[k] / sq_ar

            wh_pairs.extend([[w, h]])

        feat_h, feat_w = feat[k].shape[-2:]

        # These shifts will be the centre of each of the default boxes
        shifts_x = ((torch.arange(0, feat_w) + 0.5) / feat_w).to(torch.float32)
        shifts_y = ((torch.arange(0, feat_h) + 0.5) / feat_h).to(torch.float32)
        shift_y, shift_x = torch.meshgrid(shifts_y, shifts_x, indexing="ij")
        shift_x = shift_x.reshape(-1)
        shift_y = shift_y.reshape(-1)

        # Duplicate these shifts for as
        # many boxes(aspect ratios)
        # per position we have
        shifts = torch.stack((shift_x, shift_y) * len(wh_pairs), dim=-1).reshape(-1, 2)
        # shifts for first feature map will be (5776 x 2)

        wh_pairs = torch.as_tensor(wh_pairs)

        # Repeat the wh pairs for all positions in feature map
        wh_pairs = wh_pairs.repeat((feat_h * feat_w), 1)
        # wh_pairs for first feature map will be (5776 x 2)

        # Concat the shifts(cx cy) and wh values for all positions
        default_box = torch.cat((shifts, wh_pairs), dim=1)
        # default box for feat_1 -> (5776, 4)
        # default box for feat_2 -> (2166, 4)
        # default box for feat_3 -> (600, 4)
        # default box for feat_4 -> (150, 4)
        # default box for feat_5 -> (36, 4)
        # default box for feat_6 -> (4, 4)

        default_boxes.append(default_box)
    default_boxes = torch.cat(default_boxes, dim=0)
    # default_boxes -> (8732, 4)

    # We now duplicate these default boxes
    # for all images in the batch
    # and also convert cx,cy,w,h format of
    # default boxes to x1,y1,x2,y2
    dboxes = []
    for _ in range(feat[0].size(0)):
        dboxes_in_image = default_boxes
        # x1 = cx - 0.5 * width
        # y1 = cy - 0.5 * height
        # x2 = cx + 0.5 * width
        # y2 = cy + 0.5 * height
        dboxes_in_image = torch.cat(
            [
                (dboxes_in_image[:, :2] - 0.5 * dboxes_in_image[:, 2:]),
                (dboxes_in_image[:, :2] + 0.5 * dboxes_in_image[:, 2:]),
            ],
            -1,
        )
        dboxes.append(dboxes_in_image.to(feat[0].device))
    return dboxes


class SSD(nn.Module):
    r"""
    Main Class for SSD. Does the following steps
    to generate detections/losses.
    During initialization
    1. Load VGG Imagenet pretrained model
    2. Extract Backbone from VGG and add extra conv layers
    3. Add class prediction and bbox transformation prediction layers
    4. Initialize all conv2d layers

    During Forward Pass
    1. Get conv4_3 output
    2. Normalize and scale conv4_3 output (feat_output_1)
    3. Pass the unscaled conv4_3 to conv5_3 layers and conv layers
        replacing fc6 and fc7 of vgg (feat_output_2)
    4. Pass the conv_fc7 output to extra conv layers (feat_output_3-6)
    5. Get the classification and regression predictions for all 6 feature maps
    6. Generate default_boxes for all these feature maps(8732 x 4)
    7a. If in training assign targets for these default_boxes and
        compute localization and classification losses
    7b. If in inference mode, then do all pre-nms filtering, nms
        and then post nms filtering and return the detected boxes,
        their labels and their scores
    """
    def __init__(self, config, num_classes=21):
        super().__init__()
        self.aspect_ratios = config['aspect_ratios']

        self.scales = config['scales']
        self.scales.append(1.0)

        self.num_classes = num_classes
        self.iou_threshold = config['iou_threshold']
        self.low_score_threshold = config['low_score_threshold']
        self.neg_pos_ratio = config['neg_pos_ratio']
        self.pre_nms_topK = config['pre_nms_topK']
        self.nms_threshold = config['nms_threshold']
        self.detections_per_img = config['detections_per_img']

        # Load imagenet pretrained vgg network
        backbone = torchvision.models.vgg16(
            weights=torchvision.models.VGG16_Weights.IMAGENET1K_V1
        )

        # Get all max pool indexes to determine different stages
        max_pool_pos = [idx for idx, layer in enumerate(list(backbone.features))
                        if isinstance(layer, nn.MaxPool2d)]
        max_pool_stage_3_pos = max_pool_pos[-3]  # for vgg16 this would be 16
        max_pool_stage_4_pos = max_pool_pos[-2]  # for vgg16 this would be 23

        backbone.features[max_pool_stage_3_pos].ceil_mode = True
        # otherwise vgg conv4_3 output will be 37x37
        self.features = nn.Sequential(*backbone.features[:max_pool_stage_4_pos])
        self.scale_weight = nn.Parameter(torch.ones(512) * 20)

        ###################################
        # Conv5_3 + Conv for fc6 and fc 7 #
        ###################################
        # Conv modules replacing fc6 and fc7
        # Ideally we would copy the weights
        # but here we are just adding new layers
        # and not copying fc6 and fc7 weights by
        # subsampling
        fcs = nn.Sequential(
            nn.MaxPool2d(kernel_size=3, stride=1, padding=1),
            nn.Conv2d(in_channels=512, out_channels=1024, kernel_size=3,
                      padding=6, dilation=6),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_channels=1024, out_channels=1024, kernel_size=1),
            nn.ReLU(inplace=True),
        )
        self.conv5_3_fc = nn.Sequential(
            *backbone.features[max_pool_stage_4_pos:-1],
            fcs,
        )

        ##########################
        # Additional Conv Layers #
        ##########################
        # Modules to take from 19x19 to 10x10
        self.conv8_2 = nn.Sequential(
            nn.Conv2d(1024, 256, kernel_size=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 512, kernel_size=3, padding=1,
                      stride=2),
            nn.ReLU(inplace=True)
        )

        # Modules to take from 10x10 to 5x5
        self.conv9_2 = nn.Sequential(
            nn.Conv2d(512, 128, kernel_size=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 256, kernel_size=3, padding=1,
                      stride=2),
            nn.ReLU(inplace=True)
        )

        # Modules to take from 5x5 to 3x3
        self.conv10_2 = nn.Sequential(
            nn.Conv2d(256, 128, kernel_size=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 256, kernel_size=3),
            nn.ReLU(inplace=True)
        )

        # Modules to take from 3x3 to 1x1
        self.conv11_2 = nn.Sequential(
            nn.Conv2d(256, 128, kernel_size=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 256, kernel_size=3),
            nn.ReLU(inplace=True)
        )

        # Must match conv4_3, fcs, conv8_2, conv9_2, conv10_2, conv11_2
        out_channels = [512, 1024, 512, 256, 256, 256]

        #####################
        # Prediction Layers #
        #####################
        self.cls_heads = nn.ModuleList()
        for channels, aspect_ratio in zip(out_channels, self.aspect_ratios):
            # extra 1 is added for scale of sqrt(sk*sk+1)
            self.cls_heads.append(nn.Conv2d(channels,
                                            self.num_classes * (len(aspect_ratio)+1),
                                            kernel_size=3,
                                            padding=1))

        self.bbox_reg_heads = nn.ModuleList()
        for channels, aspect_ratio in zip(out_channels, self.aspect_ratios):
            # extra 1 is added for scale of sqrt(sk*sk+1)
            self.bbox_reg_heads.append(nn.Conv2d(channels, 4 * (len(aspect_ratio)+1),
                                                 kernel_size=3,
                                                 padding=1))

        #############################
        # Conv Layer Initialization #
        #############################
        for layer in fcs.modules():
            if isinstance(layer, nn.Conv2d):
                torch.nn.init.xavier_uniform_(layer.weight)
                if layer.bias is not None:
                    torch.nn.init.constant_(layer.bias, 0.0)

        for conv_module in [self.conv8_2, self.conv9_2, self.conv10_2, self.conv11_2]:
            for layer in conv_module.modules():
                if isinstance(layer, nn.Conv2d):
                    torch.nn.init.xavier_uniform_(layer.weight)
                    if layer.bias is not None:
                        torch.nn.init.constant_(layer.bias, 0.0)

        for module in self.cls_heads:
            torch.nn.init.xavier_uniform_(module.weight)
            if module.bias is not None:
                torch.nn.init.constant_(module.bias, 0.0)
        for module in self.bbox_reg_heads:
            torch.nn.init.xavier_uniform_(module.weight)
            if module.bias is not None:
                torch.nn.init.constant_(module.bias, 0.0)

    def compute_loss(
            self,
            targets,
            cls_logits,
            bbox_regression,
            default_boxes,
            matched_idxs,
    ):
        # Counting all the foreground default_boxes for computing N in loss equation
        num_foreground = 0
        # BBox losses for all batch images(for foreground default_boxes)
        bbox_loss = []
        # classification targets for all batch images(for ALL default_boxes)
        cls_targets = []
        for (
            targets_per_image,
            bbox_regression_per_image,
            cls_logits_per_image,
            default_boxes_per_image,
            matched_idxs_per_image,
        ) in zip(targets, bbox_regression, cls_logits, default_boxes, matched_idxs):
            # Foreground default_boxes -> matched_idx >=0
            # Background default_boxes -> matched_idx = -1
            fg_idxs_per_image = torch.where(matched_idxs_per_image >= 0)[0]
            foreground_matched_idxs_per_image = matched_idxs_per_image[
                fg_idxs_per_image
            ]
            num_foreground += foreground_matched_idxs_per_image.numel()

            # Get foreground default_boxes and their transformation predictions
            matched_gt_boxes_per_image = targets_per_image["boxes"][
                foreground_matched_idxs_per_image
            ]
            bbox_regression_per_image = bbox_regression_per_image[fg_idxs_per_image, :]
            default_boxes_per_image = default_boxes_per_image[fg_idxs_per_image, :]
            target_regression = boxes_to_transformation_targets(
                matched_gt_boxes_per_image,
                default_boxes_per_image)

            bbox_loss.append(
                torch.nn.functional.smooth_l1_loss(bbox_regression_per_image,
                                                   target_regression,
                                                   reduction='sum')
            )

            # Get classification target for ALL default_boxes
            # For all default_boxes set it as 0 first
            # Then set foreground default_boxes target as label
            # of assigned gt box
            gt_classes_target = torch.zeros(
                (cls_logits_per_image.size(0),),
                dtype=targets_per_image["labels"].dtype,
                device=targets_per_image["labels"].device,
            )
            gt_classes_target[fg_idxs_per_image] = targets_per_image["labels"][
                foreground_matched_idxs_per_image
            ]
            cls_targets.append(gt_classes_target)

        # Aggregated bbox loss and classification targets
        # for all batch images
        bbox_loss = torch.stack(bbox_loss)
        cls_targets = torch.stack(cls_targets)  # (B, 8732)

        # Calculate classification loss for ALL default_boxes
        num_classes = cls_logits.size(-1)
        cls_loss = torch.nn.functional.cross_entropy(cls_logits.view(-1, num_classes),
                                                     cls_targets.view(-1),
                                                     reduction="none").view(
            cls_targets.size()
        )

        # Hard Negative Mining
        foreground_idxs = cls_targets > 0
        # We will sample total of 3 x (number of fg default_boxes)
        # background default_boxes
        num_negative = self.neg_pos_ratio * foreground_idxs.sum(1, keepdim=True)

        # As of now cls_loss is for ALL default_boxes
        negative_loss = cls_loss.clone()
        # We want to ensure that after sorting based on loss value,
        # foreground default_boxes are never picked when choosing topK
        # highest loss indexes
        negative_loss[foreground_idxs] = -float("inf")
        values, idx = negative_loss.sort(1, descending=True)
        # Fetch those indexes which have in topK(K=num_negative) losses
        background_idxs = idx.sort(1)[1] < num_negative
        N = max(1, num_foreground)
        return {
            "bbox_regression": bbox_loss.sum() / N,
            "classification": (cls_loss[foreground_idxs].sum() +
                               cls_loss[background_idxs].sum()) / N,
        }

    def forward(self, x, targets=None):
        # Call everything till conv4_3 layers first
        conv_4_3_out = self.features(x)

        # Scale conv4_3 output using learnt norm scale
        conv_4_3_out_scaled = (self.scale_weight.view(1, -1, 1, 1) *
                               torch.nn.functional.normalize(conv_4_3_out))

        # Call conv5_3 with non_scaled conv_3 and also
        # Call additional conv layers
        conv_5_3_fc_out = self.conv5_3_fc(conv_4_3_out)
        conv8_2_out = self.conv8_2(conv_5_3_fc_out)
        conv9_2_out = self.conv9_2(conv8_2_out)
        conv10_2_out = self.conv10_2(conv9_2_out)
        conv11_2_out = self.conv11_2(conv10_2_out)

        # Feature maps for predictions
        outputs = [
            conv_4_3_out_scaled,  # 38 x 38
            conv_5_3_fc_out,  # 19 x 19
            conv8_2_out,  # 10 x 10
            conv9_2_out,  # 5 x 5
            conv10_2_out,  # 3 x 3
            conv11_2_out,   # 1 x 1
        ]

        # Classification and bbox regression for all feature maps
        cls_logits = []
        bbox_reg_deltas = []
        for i, features in enumerate(outputs):
            cls_feat_i = self.cls_heads[i](features)
            bbox_reg_feat_i = self.bbox_reg_heads[i](features)

            # Cls output from (B, A * num_classes, H, W) to (B, HWA, num_classes).
            N, _, H, W = cls_feat_i.shape
            cls_feat_i = cls_feat_i.view(N, -1, self.num_classes, H, W)
            # (B, A, num_classes, H, W)
            cls_feat_i = cls_feat_i.permute(0, 3, 4, 1, 2)  # (B, H, W, A, num_classes)
            cls_feat_i = cls_feat_i.reshape(N, -1, self.num_classes)
            # (B, HWA, num_classes)
            cls_logits.append(cls_feat_i)

            # Permute bbox reg output from (B, A * 4, H, W) to (B, HWA, 4).
            N, _, H, W = bbox_reg_feat_i.shape
            bbox_reg_feat_i = bbox_reg_feat_i.view(N, -1, 4, H, W)  # (B, A, 4, H, W)
            bbox_reg_feat_i = bbox_reg_feat_i.permute(0, 3, 4, 1, 2)  # (B, H, W, A, 4)
            bbox_reg_feat_i = bbox_reg_feat_i.reshape(N, -1, 4)  # Size=(B, HWA, 4)
            bbox_reg_deltas.append(bbox_reg_feat_i)

        # Concat cls logits and bbox regression predictions for all feature maps
        cls_logits = torch.cat(cls_logits, dim=1)  # (B, 8732, num_classes)
        bbox_reg_deltas = torch.cat(bbox_reg_deltas, dim=1)  # (B, 8732, 4)

        # Generate default_boxes for all feature maps
        default_boxes = generate_default_boxes(outputs, self.aspect_ratios, self.scales)
        # default_boxes -> List[Tensor of shape 8732 x 4]
        # len(default_boxes) = Batch size

        losses = {}
        detections = []
        if self.training:
            # List to hold for each image, which default box
            # is assigned to with gt box if any
            # or unassigned(background)
            matched_idxs = []
            for default_boxes_per_image, targets_per_image in zip(default_boxes,
                                                                  targets):
                if targets_per_image["boxes"].numel() == 0:
                    matched_idxs.append(
                        torch.full(
                            (default_boxes_per_image.size(0),), -1,
                            dtype=torch.int64,
                            device=default_boxes_per_image.device
                        )
                    )
                    continue
                iou_matrix = get_iou(targets_per_image["boxes"],
                                     default_boxes_per_image)
                # For each default box find best ground truth box
                matched_vals, matches = iou_matrix.max(dim=0)
                # matches -> [8732]

                # Update index of match for all default_boxes which
                # have maximum iou with a gt box < low threshold
                # as -1
                # This allows selecting foreground boxes as match index >= 0
                below_low_threshold = matched_vals < self.iou_threshold
                matches[below_low_threshold] = -1

                # We want to also assign the best default box for every gt
                # as foreground
                # So first find the best default box for every gt
                _, highest_quality_pred_foreach_gt = iou_matrix.max(dim=1)
                # Update the best matching gt index for these best default_boxes
                # as 0, 1, 2, ...., len(gt)-1
                matches[highest_quality_pred_foreach_gt] = torch.arange(
                    highest_quality_pred_foreach_gt.size(0), dtype=torch.int64,
                    device=highest_quality_pred_foreach_gt.device
                )
                matched_idxs.append(matches)
            losses = self.compute_loss(targets, cls_logits, bbox_reg_deltas,
                                       default_boxes, matched_idxs)
        else:
            # For test time we do the following:
            # 1. Convert default_boxes to boxes using predicted bbox regression deltas
            # 2. Low score filtering
            # 3. Pre-NMS TopK filtering
            # 4. NMS
            # 5. Post NMS TopK Filtering
            cls_scores = torch.nn.functional.softmax(cls_logits, dim=-1)
            num_classes = cls_scores.size(-1)

            for bbox_deltas_i, cls_scores_i, default_boxes_i in zip(bbox_reg_deltas,
                                                                    cls_scores,
                                                                    default_boxes):
                boxes = apply_regression_pred_to_default_boxes(bbox_deltas_i,
                                                               default_boxes_i)
                # Ensure all values are between 0-1
                boxes.clamp_(min=0., max=1.)

                pred_boxes = []
                pred_scores = []
                pred_labels = []
                # Class wise filtering
                for label in range(1, num_classes):
                    score = cls_scores_i[:, label]

                    # Remove low scoring boxes of this class
                    keep_idxs = score > self.low_score_threshold
                    score = score[keep_idxs]
                    box = boxes[keep_idxs]

                    # keep only topk scoring predictions of this class
                    score, top_k_idxs = score.topk(min(self.pre_nms_topK, len(score)))
                    box = box[top_k_idxs]

                    pred_boxes.append(box)
                    pred_scores.append(score)
                    pred_labels.append(torch.full_like(score, fill_value=label,
                                                       dtype=torch.int64,
                                                       device=cls_scores.device))

                pred_boxes = torch.cat(pred_boxes, dim=0)
                pred_scores = torch.cat(pred_scores, dim=0)
                pred_labels = torch.cat(pred_labels, dim=0)

                # Class wise NMS
                keep_mask = torch.zeros_like(pred_scores, dtype=torch.bool)
                for class_id in torch.unique(pred_labels):
                    curr_indices = torch.where(pred_labels == class_id)[0]
                    curr_keep_idxs = torch.ops.torchvision.nms(pred_boxes[curr_indices],
                                                               pred_scores[curr_indices],
                                                               self.nms_threshold)
                    keep_mask[curr_indices[curr_keep_idxs]] = True
                keep_indices = torch.where(keep_mask)[0]
                post_nms_keep_indices = keep_indices[pred_scores[keep_indices].sort(
                    descending=True)[1]]
                keep = post_nms_keep_indices[:self.detections_per_img]
                pred_boxes, pred_scores, pred_labels = (pred_boxes[keep],
                                                        pred_scores[keep],
                                                        pred_labels[keep])

                detections.append(
                    {
                        "boxes": pred_boxes,
                        "scores": pred_scores,
                        "labels": pred_labels,
                    }
                )
        return losses, detections

**TRAINING**

In [8]:
train_dataset = VOCDataset(
    split="train",
    dataset_path="merged_dataset",
    im_size=300
)

print("\nDataset size:", len(train_dataset))

image, target, filename = train_dataset[0]

print("Image shape:", image.shape)
print("Boxes:", target["bboxes"].shape)
print("Labels:", target["labels"])
print("File:", filename)
print("Dataset classes:", len(train_dataset.classes))
print("Model classes:", config["dataset_params"]["num_classes"])


Classes:
0: background
1: authentic
2: counterfeit

train images: 3639

Dataset size: 3639
Image shape: torch.Size([3, 300, 300])
Boxes: torch.Size([1, 4])
Labels: tensor([1])
File: merged_dataset/train/ds1_IMG_20251016_153853_128_jpg.rf.fb6ad89e332c2b441fb5b170f425afa0.jpg
Dataset classes: 3
Model classes: 3


In [9]:
import os
import random
import time
import numpy as np
import torch

from tqdm import tqdm
from torch.utils.data import DataLoader
from torch.optim.lr_scheduler import MultiStepLR

if torch.cuda.is_available():
    device = torch.device("cuda:0")
    gpu_count = torch.cuda.device_count()

    print(f"CUDA available: {torch.cuda.is_available()}")
    print(f"GPU count: {gpu_count}")

    for i in range(gpu_count):
        print(f"{i}: {torch.cuda.get_device_name(i)}")

else:
    device = torch.device("cpu")
    gpu_count = 0
    print("CUDA available: False")

print(f"Using device: {device}")

def collate_function(data):
    return tuple(zip(*data))


def train(config):

    dataset_config = config["dataset_params"]
    train_config = config["train_params"]

    seed = train_config["seed"]

    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    os.makedirs(
        train_config["task_name"],
        exist_ok=True
    )

    dataset_path = dataset_config["dataset_path"]

    print("Dataset:", dataset_path)

    train_dataset = VOCDataset(
        split="train",
        dataset_path=dataset_path,
        im_size=dataset_config["im_size"]
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=train_config["batch_size"],
        shuffle=True,
        collate_fn=collate_function,
        num_workers=2,
        pin_memory=torch.cuda.is_available()
    )

    print("Training images:", len(train_dataset))

    model = SSD(
        config=config["model_params"],
        num_classes=dataset_config["num_classes"]
    )

    model.to(device)
    model.train()

    checkpoint_path = os.path.join(
        train_config["task_name"],
        train_config["ckpt_name"]
    )

    if os.path.exists(checkpoint_path):

        print("Loading checkpoint:", checkpoint_path)

        model.load_state_dict(
            torch.load(
                checkpoint_path,
                map_location=device
            )
        )

    optimizer = torch.optim.SGD(
        params=model.parameters(),
        lr=train_config["lr"],
        weight_decay=5e-4,
        momentum=0.9
    )

    lr_scheduler = MultiStepLR(
        optimizer,
        milestones=train_config["lr_steps"],
        gamma=0.5
    )

    acc_steps = train_config["acc_steps"]
    num_epochs = train_config["num_epochs"]
    log_steps = train_config["log_steps"]

    steps = 0

    training_start = time.time()

    for epoch in range(num_epochs):

        model.train()

        ssd_classification_losses = []
        ssd_localization_losses = []

        progress_bar = tqdm(
            train_loader,
            desc=f"Epoch {epoch + 1}/{num_epochs}"
        )

        optimizer.zero_grad()

        for idx, (ims, targets, _) in enumerate(progress_bar):

            for target in targets:

                target["boxes"] = (
                    target["bboxes"]
                    .float()
                    .to(device)
                )

                del target["bboxes"]

                target["labels"] = (
                    target["labels"]
                    .long()
                    .to(device)
                )

            images = torch.stack(
                [
                    im.float().to(device)
                    for im in ims
                ],
                dim=0
            )

            batch_losses, _ = model(
                images,
                targets
            )

            classification_loss = (
                batch_losses["classification"]
            )

            localization_loss = (
                batch_losses["bbox_regression"]
            )

            loss = (
                classification_loss
                + localization_loss
            )

            if torch.isnan(loss):
                print("Loss became NaN. Stopping training.")
                return model

            ssd_classification_losses.append(
                classification_loss.item()
            )

            ssd_localization_losses.append(
                localization_loss.item()
            )

            loss = loss / acc_steps

            loss.backward()

            if (idx + 1) % acc_steps == 0:

                optimizer.step()
                optimizer.zero_grad()

            if steps % log_steps == 0:

                avg_cls = np.mean(
                    ssd_classification_losses
                )

                avg_loc = np.mean(
                    ssd_localization_losses
                )

                progress_bar.set_postfix(
                    classification=f"{avg_cls:.4f}",
                    localization=f"{avg_loc:.4f}",
                    loss=f"{avg_cls + avg_loc:.4f}"
                )

            steps += 1

        if len(train_loader) % acc_steps != 0:

            optimizer.step()
            optimizer.zero_grad()

        lr_scheduler.step()

        epoch_cls = np.mean(
            ssd_classification_losses
        )

        epoch_loc = np.mean(
            ssd_localization_losses
        )

        epoch_loss = epoch_cls + epoch_loc

        print(
            f"\nEpoch {epoch + 1}/{num_epochs}"
        )

        print(
            f"Classification Loss: {epoch_cls:.4f}"
        )

        print(
            f"Localization Loss: {epoch_loc:.4f}"
        )

        print(
            f"Total Loss: {epoch_loss:.4f}"
        )

        print(
            f"Learning Rate: {optimizer.param_groups[0]['lr']:.6f}"
        )

        torch.save(
            model.state_dict(),
            checkpoint_path
        )

        print(
            f"Checkpoint saved: {checkpoint_path}"
        )

    training_time = time.time() - training_start

    print("\nTraining complete.")

    print(
        f"Total training time: "
        f"{training_time / 60:.2f} minutes"
    )

    return model


model = train(config)

CUDA available: True
GPU count: 2
0: Tesla T4
1: Tesla T4
Using device: cuda:0
Dataset: merged_dataset

Classes:
0: background
1: authentic
2: counterfeit

train images: 3639
Training images: 3639
Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100%|██████████| 528M/528M [00:02<00:00, 222MB/s]
Epoch 1/2: 100%|██████████| 455/455 [04:03<00:00,  1.87it/s, classification=3.3183, localization=1.7120, loss=5.0304]



Epoch 1/2
Classification Loss: 3.1711
Localization Loss: 1.6181
Total Loss: 4.7891
Learning Rate: 0.001000
Checkpoint saved: voc/ssd_final_aimedisina.pth


Epoch 2/2: 100%|██████████| 455/455 [04:11<00:00,  1.81it/s, classification=1.7296, localization=0.6774, loss=2.4069]



Epoch 2/2
Classification Loss: 1.7205
Localization Loss: 0.6728
Total Loss: 2.3933
Learning Rate: 0.001000
Checkpoint saved: voc/ssd_final_aimedisina.pth

Training complete.
Total training time: 8.26 minutes


**Evaluation**

In [10]:
import os
import random
import time
import numpy as np
import cv2
import torch
import pandas as pd
import matplotlib.pyplot as plt

from tqdm import tqdm
from torch.utils.data import DataLoader
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix

if torch.cuda.is_available():
    device = torch.device("cuda:0")
    gpu_count = torch.cuda.device_count()
    print(f"CUDA available: {torch.cuda.is_available()}")
    print(f"GPU count: {gpu_count}")
    for i in range(gpu_count):
        print(f"{i}: {torch.cuda.get_device_name(i)}")
else:
    device = torch.device("cpu")
    gpu_count = 0
    print("CUDA available: False")

print(f"Using device: {device}")


def get_iou(det, gt):
    det_x1, det_y1, det_x2, det_y2 = det
    gt_x1, gt_y1, gt_x2, gt_y2 = gt

    x_left = max(det_x1, gt_x1)
    y_top = max(det_y1, gt_y1)
    x_right = min(det_x2, gt_x2)
    y_bottom = min(det_y2, gt_y2)

    if x_right <= x_left or y_bottom <= y_top:
        return 0.0

    intersection = (
        (x_right - x_left) *
        (y_bottom - y_top)
    )

    det_area = max(0, det_x2 - det_x1) * max(0, det_y2 - det_y1)
    gt_area = max(0, gt_x2 - gt_x1) * max(0, gt_y2 - gt_y1)

    union = det_area + gt_area - intersection + 1e-6

    return intersection / union


def compute_map(det_boxes, gt_boxes, iou_threshold=0.5, method="area", difficult=None):
    gt_labels = set()

    for image_gt in gt_boxes:
        gt_labels.update(image_gt.keys())

    gt_labels = sorted(gt_labels)

    all_aps = {}
    aps = []

    for label in gt_labels:
        cls_dets = []

        for im_idx, im_dets in enumerate(det_boxes):
            if label not in im_dets:
                continue

            for detection in im_dets[label]:
                cls_dets.append([im_idx, detection])

        cls_dets = sorted(cls_dets, key=lambda x: -x[1][-1])

        gt_matched = []

        for image_gt in gt_boxes:
            boxes = image_gt.get(label, [])
            gt_matched.append([False] * len(boxes))

        num_gts = sum(
            len(image_gt.get(label, []))
            for image_gt in gt_boxes
        )

        num_difficults = 0

        if difficult is not None:
            num_difficults = sum(
                sum(image_diff.get(label, []))
                for image_diff in difficult
            )

        tp = np.zeros(len(cls_dets), dtype=np.float32)
        fp = np.zeros(len(cls_dets), dtype=np.float32)

        for det_idx, (im_idx, det_pred) in enumerate(cls_dets):
            im_gts = gt_boxes[im_idx].get(label, [])

            im_difficult = (
                difficult[im_idx].get(label, [])
                if difficult is not None
                else [0] * len(im_gts)
            )

            if len(im_gts) == 0:
                fp[det_idx] = 1
                continue

            max_iou = -1
            max_gt_idx = -1

            for gt_idx, gt_box in enumerate(im_gts):
                iou = get_iou(det_pred[:-1], gt_box)

                if iou > max_iou:
                    max_iou = iou
                    max_gt_idx = gt_idx

            if max_iou >= iou_threshold:
                if im_difficult[max_gt_idx]:
                    continue

                if not gt_matched[im_idx][max_gt_idx]:
                    gt_matched[im_idx][max_gt_idx] = True
                    tp[det_idx] = 1
                else:
                    fp[det_idx] = 1
            else:
                fp[det_idx] = 1

        tp = np.cumsum(tp)
        fp = np.cumsum(fp)

        eps = np.finfo(np.float32).eps

        valid_gts = max(num_gts - num_difficults, 0)

        recalls = tp / np.maximum(valid_gts, eps)
        precisions = tp / np.maximum(tp + fp, eps)

        if method == "area":
            recalls = np.concatenate(([0.0], recalls, [1.0]))
            precisions = np.concatenate(([0.0], precisions, [0.0]))

            for i in range(precisions.size - 1, 0, -1):
                precisions[i - 1] = max(
                    precisions[i - 1],
                    precisions[i]
                )

            indices = np.where(
                recalls[1:] != recalls[:-1]
            )[0]

            ap = np.sum(
                (recalls[indices + 1] - recalls[indices]) *
                precisions[indices + 1]
            )

        elif method == "interp":
            ap = 0.0

            for recall_point in np.arange(0, 1.01, 0.1):
                values = precisions[recalls >= recall_point]

                if values.size > 0:
                    ap += values.max()

            ap /= 11.0

        else:
            raise ValueError("method must be 'area' or 'interp'")

        if num_gts > 0:
            aps.append(ap)
            all_aps[label] = ap
        else:
            all_aps[label] = np.nan

    if len(aps) > 0:
        mean_ap = float(np.mean(aps))
    else:
        mean_ap = 0.0

    return mean_ap, all_aps


def load_model_and_dataset(config):
    dataset_config = config["dataset_params"]
    model_config = config["model_params"]
    train_config = config["train_params"]

    dataset_path = dataset_config["dataset_path"]

    test_dataset = VOCDataset(
        split="test",
        dataset_path=dataset_path,
        im_size=dataset_config["im_size"]
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=1,
        shuffle=False,
        num_workers=2,
        pin_memory=torch.cuda.is_available()
    )

    model = SSD(
        config=model_config,
        num_classes=dataset_config["num_classes"]
    )

    model.to(device)

    checkpoint_path = os.path.join(
        train_config["task_name"],
        train_config["ckpt_name"]
    )

    if not os.path.exists(checkpoint_path):
        raise FileNotFoundError(
            f"Checkpoint not found:\n{checkpoint_path}"
        )

    print("Loading checkpoint:", checkpoint_path)

    checkpoint = torch.load(
        checkpoint_path,
        map_location=device
    )

    model.load_state_dict(checkpoint)
    model.eval()

    print("Test images:", len(test_dataset))
    print("Number of classes:", dataset_config["num_classes"])

    return model, test_dataset, test_loader


def infer_samples(config, num_samples=5):
    os.makedirs("samples", exist_ok=True)

    model, voc, _ = load_model_and_dataset(config)

    confidence_threshold = (
        config["train_params"]["infer_conf_threshold"]
    )

    model.low_score_threshold = confidence_threshold

    if len(voc) == 0:
        print("Test dataset is empty.")
        return

    num_samples = min(num_samples, len(voc))

    sample_indices = random.sample(
        range(len(voc)),
        num_samples
    )

    for i, dataset_idx in enumerate(
        tqdm(sample_indices, desc="Generating samples")
    ):
        im_tensor, target, fname = voc[dataset_idx]

        with torch.no_grad():
            _, detections = model(
                im_tensor.unsqueeze(0).to(device),
                [target]
            )

        original = cv2.imread(fname)

        if original is None:
            print(f"Could not read image: {fname}")
            continue

        h, w = original.shape[:2]

        gt_image = original.copy()
        prediction_image = original.copy()

        for idx, box in enumerate(target["bboxes"]):
            x1, y1, x2, y2 = (
                box.detach().cpu().numpy()
            )

            x1 = int(w * x1)
            y1 = int(h * y1)
            x2 = int(w * x2)
            y2 = int(h * y2)

            label_id = (
                target["labels"][idx]
                .detach()
                .cpu()
                .item()
            )

            label_name = voc.idx2label[label_id]

            cv2.rectangle(
                gt_image,
                (x1, y1),
                (x2, y2),
                (0, 255, 0),
                2
            )

            cv2.putText(
                gt_image,
                label_name,
                (x1 + 5, y1 + 15),
                cv2.FONT_HERSHEY_PLAIN,
                1,
                (0, 0, 0),
                1
            )

        cv2.imwrite(
            f"samples/output_ssd_gt_{i}.jpg",
            gt_image
        )

        boxes = detections[0]["boxes"]
        labels = detections[0]["labels"]
        scores = detections[0]["scores"]

        for idx, box in enumerate(boxes):
            x1, y1, x2, y2 = (
                box.detach().cpu().numpy()
            )

            x1 = int(w * x1)
            y1 = int(h * y1)
            x2 = int(w * x2)
            y2 = int(h * y2)

            label_id = (
                labels[idx]
                .detach()
                .cpu()
                .item()
            )

            score = (
                scores[idx]
                .detach()
                .cpu()
                .item()
            )

            if label_id not in voc.idx2label:
                continue

            label_name = voc.idx2label[label_id]

            text = f"{label_name}: {score:.2f}"

            cv2.rectangle(
                prediction_image,
                (x1, y1),
                (x2, y2),
                (0, 0, 255),
                2
            )

            cv2.putText(
                prediction_image,
                text,
                (x1 + 5, y1 + 15),
                cv2.FONT_HERSHEY_PLAIN,
                1,
                (0, 0, 0),
                1
            )

        cv2.imwrite(
            f"samples/output_ssd_{i}.jpg",
            prediction_image
        )

    print("Done detecting.")
    print("Saved images to: samples/")


def evaluate_model(config, training_time_seconds=None):
    model, voc, test_loader = load_model_and_dataset(config)

    model.low_score_threshold = (
        config["train_params"]["infer_conf_threshold"]
    )

    iou_threshold = 0.50

    gts = []
    preds = []
    difficults = []

    y_true = []
    y_pred = []

    class_ids = sorted(
        [idx for idx in voc.idx2label.keys() if idx != 0]
    )

    evaluation_start = time.time()

    for im_tensor, target, fname in tqdm(
        test_loader,
        desc="Evaluating"
    ):
        im_tensor = im_tensor.float().to(device)

        target_bboxes = (
            target["bboxes"][0]
            .float()
            .to(device)
        )

        target_labels = (
            target["labels"][0]
            .long()
            .to(device)
        )

        difficult = (
            target["difficult"][0]
            .long()
            .to(device)
        )

        with torch.no_grad():
            _, detections = model(im_tensor)

        boxes = detections[0]["boxes"]
        labels = detections[0]["labels"]
        scores = detections[0]["scores"]

        pred_boxes = {}
        gt_boxes = {}
        difficult_boxes = {}

        for label_name in voc.label2idx:
            pred_boxes[label_name] = []
            gt_boxes[label_name] = []
            difficult_boxes[label_name] = []

        prediction_data = []

        for idx, box in enumerate(boxes):
            label_id = (
                labels[idx]
                .detach()
                .cpu()
                .item()
            )

            if label_id not in voc.idx2label:
                continue

            score = (
                scores[idx]
                .detach()
                .cpu()
                .item()
            )

            box_np = (
                box.detach()
                .cpu()
                .numpy()
            )

            label_name = voc.idx2label[label_id]

            pred_boxes[label_name].append([
                box_np[0],
                box_np[1],
                box_np[2],
                box_np[3],
                score
            ])

            prediction_data.append({
                "box": box_np,
                "label_id": label_id,
                "score": score
            })

        gt_data = []

        for idx, box in enumerate(target_bboxes):
            label_id = (
                target_labels[idx]
                .detach()
                .cpu()
                .item()
            )

            if label_id not in voc.idx2label:
                continue

            box_np = (
                box.detach()
                .cpu()
                .numpy()
            )

            label_name = voc.idx2label[label_id]

            gt_boxes[label_name].append([
                box_np[0],
                box_np[1],
                box_np[2],
                box_np[3]
            ])

            difficult_boxes[label_name].append(
                difficult[idx]
                .detach()
                .cpu()
                .item()
            )

            gt_data.append({
                "box": box_np,
                "label_id": label_id
            })

        prediction_data.sort(
            key=lambda x: x["score"],
            reverse=True
        )

        matched_gt = set()

        for prediction in prediction_data:
            best_iou = 0.0
            best_gt_idx = -1

            for gt_idx, gt in enumerate(gt_data):
                if gt_idx in matched_gt:
                    continue

                iou = get_iou(
                    prediction["box"],
                    gt["box"]
                )

                if iou > best_iou:
                    best_iou = iou
                    best_gt_idx = gt_idx

            if (
                best_gt_idx >= 0
                and best_iou >= iou_threshold
            ):
                gt = gt_data[best_gt_idx]

                matched_gt.add(best_gt_idx)

                y_true.append(gt["label_id"])
                y_pred.append(prediction["label_id"])

            else:
                y_true.append(0)
                y_pred.append(prediction["label_id"])

        for gt_idx, gt in enumerate(gt_data):
            if gt_idx not in matched_gt:
                y_true.append(gt["label_id"])
                y_pred.append(0)

        gts.append(gt_boxes)
        preds.append(pred_boxes)
        difficults.append(difficult_boxes)

    evaluation_time = time.time() - evaluation_start

    mean_ap, all_aps = compute_map(
        preds,
        gts,
        iou_threshold=0.5,
        method="area",
        difficult=difficults
    )

    accuracy = (
        np.mean(
            np.array(y_true) == np.array(y_pred)
        )
        if len(y_true) > 0
        else 0.0
    )

    precision = precision_score(
        y_true,
        y_pred,
        labels=class_ids,
        average="macro",
        zero_division=0
    )

    recall = recall_score(
        y_true,
        y_pred,
        labels=class_ids,
        average="macro",
        zero_division=0
    )

    f1 = f1_score(
        y_true,
        y_pred,
        labels=class_ids,
        average="macro",
        zero_division=0
    )

    if training_time_seconds is not None:
        hours = int(training_time_seconds // 3600)
        minutes = int((training_time_seconds % 3600) // 60)
        seconds = int(training_time_seconds % 60)

        training_time = (
            f"{hours:02d}:{minutes:02d}:{seconds:02d}"
        )
    else:
        training_time = "N/A"

    labels_for_cm = [0] + class_ids

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=labels_for_cm
    )

    class_names = (
        ["Background"] +
        [
            voc.idx2label[idx]
            for idx in class_ids
        ]
    )

    cm_df = pd.DataFrame(
        cm,
        index=class_names,
        columns=class_names
    )

    print("\n" + "=" * 70)
    print("CONFUSION MATRIX")
    print("=" * 70)
    print(cm_df)

    plt.figure(
        figsize=(8, 6)
    )

    plt.imshow(cm, cmap="Blues")

    plt.title("SSD Confusion Matrix")
    plt.colorbar()

    plt.xticks(
        range(len(class_names)),
        class_names,
        rotation=45,
        ha="right"
    )

    plt.yticks(
        range(len(class_names)),
        class_names
    )

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(
                j,
                i,
                str(cm[i, j]),
                ha="center",
                va="center"
            )

    plt.xlabel("Predicted Class")
    plt.ylabel("Actual Class")
    plt.tight_layout()

    plt.savefig(
        "confusion_matrix.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.close()

    results_table = pd.DataFrame({
        "Metric": [
            "Accuracy",
            "Precision",
            "Recall",
            "F1-Score",
            "mAP@0.5",
            "Training Time"
        ],
        "Value": [
            f"{accuracy * 100:.2f}%",
            f"{precision * 100:.2f}%",
            f"{recall * 100:.2f}%",
            f"{f1 * 100:.2f}%",
            f"{mean_ap * 100:.2f}%",
            training_time
        ]
    })

    print("\n" + "=" * 70)
    print("FINAL EVALUATION RESULTS")
    print("=" * 70)
    print(results_table.to_string(index=False))

    print("\n" + "=" * 70)
    print("CLASS-WISE AVERAGE PRECISION")
    print("=" * 70)

    ap_table = []

    for label_id, label_name in voc.idx2label.items():
        if label_id == 0:
            continue

        ap = all_aps.get(
            label_name,
            np.nan
        )

        ap_table.append({
            "Class": label_name,
            "AP@0.5": (
                "N/A"
                if np.isnan(ap)
                else f"{ap * 100:.2f}%"
            )
        })

    print(
        pd.DataFrame(ap_table).to_string(
            index=False
        )
    )

    print("\nConfusion matrix saved to: confusion_matrix.png")
    print(f"Evaluation time: {evaluation_time:.2f} seconds")

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1_score": f1,
        "map50": mean_ap,
        "training_time_seconds": training_time_seconds,
        "confusion_matrix": cm,
        "class_ap": all_aps
    }


config["dataset_params"]["num_classes"] = 3

infer_samples(
    config,
    num_samples=5
)

results = evaluate_model(
    config,
    training_time_seconds=None
)

CUDA available: True
GPU count: 2
0: Tesla T4
1: Tesla T4
Using device: cuda:0

Classes:
0: background
1: authentic
2: counterfeit

test images: 389
Loading checkpoint: voc/ssd_final_aimedisina.pth
Test images: 389
Number of classes: 3


Generating samples: 100%|██████████| 5/5 [00:00<00:00, 10.68it/s]


Done detecting.
Saved images to: samples/

Classes:
0: background
1: authentic
2: counterfeit

test images: 389
Loading checkpoint: voc/ssd_final_aimedisina.pth
Test images: 389
Number of classes: 3


Evaluating: 100%|██████████| 389/389 [00:08<00:00, 48.10it/s]



CONFUSION MATRIX
             Background  authentic  counterfeit
Background            0         37           99
authentic             7         53            1
counterfeit           4        227          103

FINAL EVALUATION RESULTS
       Metric  Value
     Accuracy 29.38%
    Precision 33.73%
       Recall 58.86%
     F1-Score 33.20%
      mAP@0.5 65.74%
Training Time    N/A

CLASS-WISE AVERAGE PRECISION
      Class AP@0.5
  authentic 73.24%
counterfeit 58.23%

Confusion matrix saved to: confusion_matrix.png
Evaluation time: 8.09 seconds
